# Crack graph — unlabeled

Build the graph, find shortest paths per entry/exit node, enumerate all simple paths within a radius, and plot the path-length distributions.

In [ ]:
from combra import graph, data

import numpy as np
from tqdm.notebook import tqdm

# Build graph

In [ ]:
border = 30
eps = 300

_, image = data.microstructure_images()[0]

(entry_nodes,
 exit_nodes,
 img_contours_o,
 img_preprocessed_final,
 cnts,
 nodes_metadata) = graph.preprocess_graph_image(
    image, border=border, disk=5, entry_ellps_w=5, exit_ellps_w=5, r=4)

# create_crack_graph now seeds 'weight'/'path_len_pixels' from the edge length,
# so no manual weight loop is needed.
g, img_contours = graph.create_crack_graph(
    img_preprocessed_final.shape, cnts, nodes_metadata, eps=eps)

In [ ]:
graph.graph_plot(g, img_contours_o)

# Shortest path per entry / exit node

In [ ]:
paths = graph.shortest_energy_paths_all_pairs(
    g, cnts, nodes_metadata, entry_nodes, exit_nodes, k=1)

shortest_entry = graph.shortest_paths_per_endpoint(paths, by='entry', k=1)
shortest_exit = graph.shortest_paths_per_endpoint(paths, by='exit', k=1)

In [ ]:
graph.plot_paths(g, shortest_entry, img_contours_o, border=border)

# Graph size

In [ ]:
mean_degree = np.array(g.degree)[:, 1].mean()
n, k = len(entry_nodes), len(exit_nodes)

print(f'entry n={n}, exit k={k}, nodes t={len(g.nodes)}, '
      f'edges={len(g.edges)}, mean degree p={mean_degree:.2f}')
print(f'entry/exit pairs={n * k}')

# All simple paths

In [ ]:
# pair every node with nodes within `epsilon` px and enumerate simple paths
df = graph.all_simple_paths_within_radius(g, epsilon=100, cutoff=10, workers=23)

# Path-length distributions

In [ ]:
save = False

# all paths, pixel length
graph.plot_paths_dist(df['path_len_pixels'], 'entry_exit_paths_all_pixels.jpg', save=save)

# per-exit-node, pixel length
for i, exit_node in tqdm(enumerate(exit_nodes)):
    d = df[df['exit_node'] == exit_node]['path_len_pixels']
    graph.plot_paths_dist(d, f'entry_exit_paths_{i}_pixels.jpg',
                          title=f'exit node {exit_node}', save=save)

# all paths, edge length
graph.plot_paths_dist(df['path_len_edges'], 'entry_exit_paths_all_edges.jpg', save=save, bins=30)

# per-exit-node, edge length
for i, exit_node in tqdm(enumerate(exit_nodes)):
    d = df[df['exit_node'] == exit_node]['path_len_edges']
    graph.plot_paths_dist(d, f'entry_exit_paths_{i}_edges.jpg',
                          title=f'exit node {exit_node}', save=save, bins=30)